In [ ]:
import os
import pathlib as pl
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

import flopy 
from flopy.utils.triangle import Triangle
import shapely

def shapely_linestring2_coords_list(linestring,decimals=2):
    # decimals=number of decimals in output coordinates
    xy=linestring.xy
    out=np.zeros((len(xy[0]),2))
    out[:,0]=np.around(xy[0],decimals)
    out[:,1]=np.around(xy[1],decimals)
    return out

def ele_gdf(tri):
    polys=[]
    for e in tri.ele:
        verts=[]
        for i in range(1,4):
            verts.append((tri.node[e[i]]['x'],tri.node[e[i]]['y']))
        verts.append((tri.node[e[1]]['x'],tri.node[e[1]]['y']))
        polys.append(shapely.Polygon(verts))
    gdf=gpd.GeoDataFrame({'ele':tri.ele['icell'],'geometry':polys})
    return gdf




In [ ]:
model_poly=gpd.read_file('Polygon.shp')
# shp=model_poly.loc[0,'geometry']
# new_coords=[]
# for c in shp.boundary.coords:
#     new_coords.append((round(c[0]),round(c[1])))
# model_poly.loc[0,'geometry']=shapely.geometry.Polygon(new_coords)
# model_poly.to_file('Polygon.shp')    

model_poly.plot()

In [ ]:
lines=gpd.read_file('lines1.shp')
# for i in lines.index:
#     shp=lines.loc[i,'geometry']
#     new_coords=[]
#     for c in shp.coords:
#         new_coords.append((round(c[0]),round(c[1])))
#     lines.loc[i,'geometry']=shapely.geometry.LineString(new_coords)    
# lines.to_file('Lines1.shp')    
lines.plot()

In [ ]:
from flopy.utils.geospatial_utils import GeoSpatialUtil

for linestring in lines.geometry:
    geom = GeoSpatialUtil(linestring, shapetype="LineString")
    print(geom.points)

In [ ]:
points=gpd.read_file('points.shp')
points.plot()

refinement_zone=gpd.read_file('Polygon2.shp')
refinement_zone.plot()

In [ ]:
wellpts=np.zeros((len(points.index),2))
for i in points.index:
    wellpts[i,0]=points.loc[i,'geometry'].x
    wellpts[i,1]=points.loc[i,'geometry'].y

tri_ws = pl.Path("./tri_ws")
tri_ws.mkdir(exist_ok=True)
tri = Triangle(maximum_area=100000, angle=25, nodes=wellpts, model_ws="./tri_ws")
tri.add_polygon(model_poly.iloc[0]['geometry'])

for line in lines.geometry:
    # tri.add_linestring(shapely_linestring2_coords_list(line,decimals=12))
    #tri.add_linestring_hayley(line)
    tri.add_linestring(line)

In [ ]:
linestring = lines.geometry[0]
geom = GeoSpatialUtil(linestring, shapetype="LineString")
linestring = geom.points

print(linestring)

In [ ]:
tri.build(verbose=True)



In [ ]:
figsize = (5, 5)
fig = plt.figure(figsize=figsize)
ax = plt.subplot(1, 1, 1, aspect="equal")
pc = tri.plot(ax=ax)
lines.plot(ax=ax)
plt.title('initial_mesh')

In [ ]:
# first refinement iteration Refine at zone defined by polygon 
ele=[]
gdf=ele_gdf(tri)
for i in gdf.index:
    if (gdf.loc[i,'geometry'].touches(refinement_zone.loc[0,'geometry']) or
        gdf.loc[i,'geometry'].within(refinement_zone.loc[0,'geometry']) or
        gdf.loc[i,'geometry'].overlaps(refinement_zone.loc[0,'geometry'])):
        ele.append(gdf.loc[i,'ele'])

tri.refine_ele(ele,20000,iteration=1,verbose=True)

fig = plt.figure(figsize=figsize)
ax = plt.subplot(1, 1, 1, aspect="equal")
pc = tri.plot(ax=ax)
plt.title('first refinement')

#second refinement at pumping wells


# second refinement around pumping bores
nds_2_refine=np.array(wellpts)
nd_numbers=tri.node[np.logical_and(np.isin(tri.node['x'],nds_2_refine[:,0]),np.isin(tri.node['y'],nds_2_refine[:,1]))]['ivert']
tri.refine_nds(nd_numbers,2000,iteration=2,verbose=True)
fig = plt.figure(figsize=figsize)
ax = plt.subplot(1, 1, 1, aspect="equal")
pc = tri.plot(ax=ax)
plt.title('second refinement')

In [ ]:
a = np.array([[1,1,1],[2,2,2],[3,3,3],[4,4,4],[5,5,5],[1,1,1],[2,2,2]])
print(f"{a=}")
unq, count = np.unique(a, axis=0, return_counts=True)
print(f"{unq=}")
print(f"{count=}")
# repeated_groups = unq[count > 1]
# print(f"{repeated_groups=}")

# for repeated_group in repeated_groups:
#     repeated_idx = np.argwhere(np.all(a == repeated_group, axis=1))
#     print(repeated_idx.ravel())
print ("**")
for row in a:
    idx = np.argwhere(np.all(a == row, axis=1))
    print(row, idx.ravel().min())

In [ ]:
Triangle.unique_vertices(a)